# 베이스라인 — 익일 등락 방향을 맞힐 수 있는가

입력은 gold `stock_forecast.stock_forecast__direction_training_h1`이다
(`feature_store_daily` ⨝ `labels_forward_return` h=1, 유동성 하한 적용).

## 질문

**과거만 보는 가격·거래량·뉴스·거시 피처로, 다음 거래일 종가가 오를지 내릴지**를
더미 기준선보다 낫게 맞힐 수 있는가.

## 설계

| 항목 | 값 |
|---|---|
| 타깃 | `label_up` = 익일 수익률 > 0 (0은 false) |
| 피처 | gold의 비식별·비라벨 컬럼 전부 — `label_` 접두는 **명시적으로 뺀다** |
| 분할 | 🔴 **시간 단위** walk-forward + embargo + purge (종목 단위 분할 아님) |
| 주지표 | ROC-AUC(풀링) + **일별 횡단면 ROC-AUC 평균**. PR-AUC는 유병률과 병기 |
| 대조 | `DummyClassifier(prior)` + **라벨 셔플 음성 대조** |
| 시드 | `RANDOM_STATE = 42` 고정 |

## 🔴 왜 시간 분할인가 — `analysis.md`의 "개체 단위 분할"을 이 데이터에 옮기면

임상 데이터의 개체는 환자지만 **시장 데이터에서 누수를 만드는 공유 요인은 날짜**다.
같은 날의 종목들은 시장 전체 움직임을 함께 겪는다 — 종목 단위로 나누면 test 종목과 같은 날의
train 종목이 "그날 시장이 올랐다"를 알려준다. 그래서 **날짜로 자르고**, 라벨이 다음 거래일을
보므로 경계에서 **embargo(=horizon)** 를 두고, 학습 행의 `label_trade_date`가 평가 구간에
들어가면 **purge**한다.

## 🔴 풀링 AUC를 그대로 믿지 않는 이유

풀링 AUC는 "어느 **날**이 오를지"(시장 방향)와 "그날 어느 **종목**이 더 오를지"(종목 선택)를
섞어 잰다. 거시 피처는 날짜마다 모든 종목에 같은 값이라 시장 방향만 설명한다.
**일별 횡단면 AUC**는 날짜 안에서만 순위를 비교하므로 시장 방향 효과가 빠진다 — 둘이 크게
갈리면 모델이 종목이 아니라 날짜를 맞히고 있는 것이다.

## 🔴 비-목적

- **투자 판단에 쓰지 않는다.** 거래비용·슬리피지·체결 가능성을 반영하지 않았다.
- 지표가 더미와 구분되지 않아도 **그것이 결론**이다(효율적 시장 가설의 약형과 맞는 결과).

## 데이터 이용 조건 — 🔴 원천 값을 출력하지 않는다

Polygon·FRED 모두 **원본 재배포 금지**([`market-data-keys.md`](../docs/setup/market-data-keys.md) §재배포 금지).
이 노트북은 **집계·지표만** 출력하고 행 단위 값(`head()` 등)을 표시하지 않는다.
모델 산출물도 저장소 밖(`$DATA_EXTRACT_DIR`)에만 둔다 — 트리 분할 임계값이 관측치 자체다.

## 전제

접속 대상은 `.env`의 `TRINO_HOST`·`TRINO_PORT`(기본 `trino.localtest.me:8080`).
gold가 비어 있으면 아래 첫 확인 셀이 멈춘다 — 선행은 `polygon_market`·`fred_calendar`
머티리얼라이즈 → `stock_forecast_dbt_models` 빌드다.
실행: `uv run --group notebook --group ml jupyter lab --port 8889`.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# repo 루트의 .env를 읽어 os.environ에 주입한다(노트북 cwd = notebooks/).
ENV_PATH = Path.cwd().parent / ".env"
loaded = load_dotenv(ENV_PATH)
TRINO_HOST = os.environ.get("TRINO_HOST", "trino.localtest.me")
TRINO_PORT = int(os.environ.get("TRINO_PORT", "8080"))

# 이 노트북 전체를 지배하는 상수 — 한곳에 모은다.
RANDOM_STATE = 42
N_SPLITS = 5
HORIZON_DAYS = 1  # gold 이름의 h1과 한 벌
EMBARGO_DAYS = HORIZON_DAYS  # 경계에서 버리는 거래일 수 — 라벨이 보는 미래 길이
MAX_ROWS = 3_000_000  # 호스트 메모리 상한 — 넘으면 멈추고 유니버스를 다시 줄인다
SLUG = "stock-direction-h1"  # 산출물 디렉터리 이름

GOLD = "iceberg.stock_forecast.stock_forecast__direction_training_h1"

# 🔴 gold 모델의 유동성 하한과 **같은 값**이어야 한다(정본은 모델 SQL의 `{% set %}`).
#    여기 두는 이유는 attrition 단계별 행 수를 세기 위해서뿐이고, 어긋나면
#    attrition 셀의 `최종 == gold 행 수` 검산이 실패해 드러난다.
MIN_CLOSE_PRICE = 5
MIN_AVG_VOLUME_20D = 1_000_000

print(f".env      : {ENV_PATH} ({'읽음' if loaded else '없음'})")
print(f"접속 대상 : http://{TRINO_HOST}:{TRINO_PORT}")
print(
    f"seed {RANDOM_STATE} / {N_SPLITS}-fold walk-forward / embargo {EMBARGO_DAYS}거래일"
)

In [ ]:
import pandas as pd
import trino

# 카탈로그 설정·자격증명은 서버 측에 있다. user는 인증이 아니라 쿼리 귀속 라벨이다.
conn = trino.dbapi.connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user="notebook",
    catalog="iceberg",
    http_scheme="http",
)


def query(sql: str) -> pd.DataFrame:
    """Trino에 SQL을 보내 DataFrame으로 받는다.

    fetchall()은 결과 전량을 클라이언트 메모리에 올린다.
    집계나 상한 확인 뒤에만 쓴다.
    """
    cur = conn.cursor()
    try:
        cur.execute(sql)
        rows = cur.fetchall()
        columns = [d[0] for d in cur.description]
    finally:
        cur.close()
    return pd.DataFrame(rows, columns=columns)


print("Trino", query("SELECT version() AS v").iloc[0]["v"])

# 🔴 관측 경로 생존 확인 — "테이블이 비었다"와 "스키마가 없다"를 가른다.
schemas = set(query("SHOW SCHEMAS FROM iceberg")["Schema"])
if "stock_forecast" not in schemas:
    msg = (
        "iceberg.stock_forecast 스키마가 없다 — gold가 아직 빌드되지 않았다.\n"
        f"보이는 스키마: {sorted(schemas)}\n"
        "선행: polygon_market·fred_calendar 머티리얼라이즈\n"
        "      → stock_forecast_dbt_models 빌드."
    )
    raise RuntimeError(msg)
tables = set(query("SHOW TABLES FROM iceberg.stock_forecast")["Table"])
assert "stock_forecast__direction_training_h1" in tables, f"gold 없음: {sorted(tables)}"
print(f"✅ stock_forecast 테이블 {len(tables)}개, gold 존재")

## 코호트 — attrition 표

제외 조건마다 몇 행이 빠지는지 센다(`analysis.md` §4). 검산: **최종 == gold 행 수**.


In [ ]:
ATTRITION_SQL = f"""
select
    count(*) as n_label_h1,
    count_if(l.label_is_complete) as n_complete,
    count_if(l.label_is_complete and f.ticker is not null) as n_joined,
    count_if(
        l.label_is_complete and f.close_price >= {MIN_CLOSE_PRICE}
    ) as n_price_ok,
    count_if(
        l.label_is_complete
        and f.close_price >= {MIN_CLOSE_PRICE}
        and f.avg_volume_20d >= {MIN_AVG_VOLUME_20D}
    ) as n_final
from iceberg.stock_forecast.labels_forward_return as l
left join iceberg.stock_forecast.feature_store_daily as f
    on l.ticker = f.ticker and l.trade_date = f.trade_date
where l.horizon_days = {HORIZON_DAYS}
"""
a = query(ATTRITION_SQL).iloc[0].astype(int)
n_gold = int(query(f"select count(*) as n from {GOLD}").iloc[0]["n"])

steps = [
    ("h=1 라벨 전체", a["n_label_h1"], ""),
    ("라벨 완결", a["n_complete"], "구간 끝 — 다음 거래일이 아직 없다"),
    ("피처 조인", a["n_joined"], "피처 행 없음"),
    (f"종가 ≥ {MIN_CLOSE_PRICE}", a["n_price_ok"], "페니 종목 — 호가 단위 잡음"),
    (
        f"20일 평균 거래량 ≥ {MIN_AVG_VOLUME_20D:,}",
        a["n_final"],
        "저유동 — 체결 가능성",
    ),
]
rows, prev = [], None
for name, n, why in steps:
    rows.append(
        {
            "단계": name,
            "행 수": f"{n:,}",
            "제외": "" if prev is None else f"{prev - n:,}",
            "사유": why,
        }
    )
    prev = n
display(pd.DataFrame(rows))

assert a["n_final"] == n_gold, (
    f"attrition 최종 {a['n_final']:,} != gold {n_gold:,}"
    " — 노트북 하한이 모델 SQL과 어긋났다"
)
assert 0 < n_gold <= MAX_ROWS, f"gold {n_gold:,}행 — 0이거나 상한 {MAX_ROWS:,} 초과"
print(f"✅ 최종 {n_gold:,}행 == gold 행 수")

In [ ]:
# 🔴 유일한 반출 지점. 행 단위 값은 표시하지 않는다(재배포 금지) — 집계만 출력한다.
df = query(f"select * from {GOLD}")
df["trade_date"] = pd.to_datetime(df["trade_date"])
df["label_trade_date"] = pd.to_datetime(df["label_trade_date"])
df = df.sort_values(["trade_date", "ticker"], kind="stable").reset_index(drop=True)

assert len(df) == n_gold, f"적재 {len(df):,} != gold {n_gold:,}"
assert (df["label_trade_date"] > df["trade_date"]).all(), (
    "라벨 날짜가 피처 날짜 이후가 아니다"
)

prevalence = df["label_up"].mean()
zero_share = (df["label_forward_return"] == 0).mean()
n_tickers = df["ticker"].nunique()
n_dates = df["trade_date"].nunique()
print(f"행 {len(df):,} / 종목 {n_tickers:,} / 거래일 {n_dates:,}")
print(f"메모리 {df.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
print(f"유병률(label_up) {prevalence:.4f}  — 주지표는 AUC(유병률 병기)")
print(f"수익률 정확히 0 비율 {zero_share:.4f} (false로 들어간다)")

In [ ]:
import numpy as np

ID_COLS = ["ticker", "trade_date"]
# 🔴 label_ 접두는 미래다. 피처 목록은 "전부 - 식별자 - 라벨"로 만들고 검산한다.
FEATURES = [c for c in df.columns if c not in ID_COLS and not c.startswith("label_")]
assert not [c for c in FEATURES if "label" in c], "라벨 계열 컬럼이 피처에 남았다"
assert FEATURES, "피처가 비었다"

y = df["label_up"].astype(int).to_numpy()
print(f"피처 {len(FEATURES)}개: {FEATURES}")
miss = df[FEATURES].isna().mean() * 100
display(miss[miss > 0].round(2).to_frame("결측률 %"))

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# 🔴 날짜 단위 walk-forward. 분할 대상은 행이 아니라 **고유 거래일**이다.
#    gap=EMBARGO_DAYS: train 끝과 test 시작 사이 거래일을 버린다.
dates = np.sort(df["trade_date"].unique())
tss = TimeSeriesSplit(n_splits=N_SPLITS, gap=EMBARGO_DAYS)
date_index = df["trade_date"].to_numpy()

folds = []
for k, (d_tr, d_te) in enumerate(tss.split(dates)):
    test_start = dates[d_te[0]]
    tr = np.flatnonzero(np.isin(date_index, dates[d_tr]))
    te = np.flatnonzero(np.isin(date_index, dates[d_te]))
    # purge — 라벨이 평가 구간 가격을 본 학습 행을 뺀다(embargo가 거래일 기준이라
    # 거래정지 종목은 label_trade_date가 더 멀 수 있다 — 그 행을 여기서 잡는다).
    purged = df["label_trade_date"].to_numpy()[tr] >= test_start
    tr = tr[~purged]
    assert df["trade_date"].iloc[tr].max() < df["trade_date"].iloc[te].min(), (
        f"fold {k}: 시간 역전"
    )
    assert df["label_trade_date"].iloc[tr].max() < test_start, f"fold {k}: purge 실패"
    folds.append((tr, te))
    print(
        f"fold {k}: train {len(tr):>9,} (purge {int(purged.sum()):,}) / "
        f"test {len(te):>8,}  test 거래일 {len(d_te)}"
    )
print()
print("✅ 전 fold에서 train 피처·라벨 날짜 < test 시작 — 시간 누수 없음")
print("🔴 test 거래일 수가 시장 방향 축의 유효 표본이다. 행 수가 아니다.")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def make_pipeline(model, scale):
    """결측 중앙값 대체(+표시자) → (선택) 표준화 → 모델."""
    steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        steps.append(("scale", StandardScaler()))
    prep = ColumnTransformer([("num", Pipeline(steps), FEATURES)], remainder="drop")
    return Pipeline([("prep", prep), ("model", model)])


def daily_auc(dates_te, y_te, p_te):
    """일별 횡단면 AUC의 평균 — 같은 날 안에서만 순위를 비교한다."""
    frame = pd.DataFrame({"d": dates_te, "y": y_te, "p": p_te})
    per_day = [
        roc_auc_score(g["y"], g["p"])
        for _, g in frame.groupby("d")
        if g["y"].nunique() == 2
    ]
    return float(np.mean(per_day)) if per_day else float("nan")


def evaluate(make_model, scale, y_vec, label):
    """walk-forward 교차검증. 한 번의 성공은 결론이 아니므로 fold 분산까지 본다."""
    auc, ap, prev, dauc = [], [], [], []
    for tr, te in folds:
        pipe = make_pipeline(make_model(), scale)
        pipe.fit(df.iloc[tr], y_vec[tr])
        p = pipe.predict_proba(df.iloc[te])[:, 1]
        auc.append(roc_auc_score(y_vec[te], p))
        ap.append(average_precision_score(y_vec[te], p))
        prev.append(y_vec[te].mean())
        dauc.append(daily_auc(date_index[te], y_vec[te], p))
    return {
        "모델": label,
        "ROC-AUC": f"{np.mean(auc):.4f} ± {np.std(auc):.4f}",
        "일별 AUC": f"{np.nanmean(dauc):.4f} ± {np.nanstd(dauc):.4f}",
        "PR-AUC": f"{np.mean(ap):.4f}",
        "유병률": f"{np.mean(prev):.4f}",
        "_auc": float(np.mean(auc)),
        "_dauc": float(np.nanmean(dauc)),
        "_ap": float(np.mean(ap)),
        "_prev": float(np.mean(prev)),
    }


MODELS = {
    "dummy(prior)": (lambda: DummyClassifier(strategy="prior"), False),
    "logistic": (lambda: LogisticRegression(max_iter=1000), True),
    "hgb": (lambda: HistGradientBoostingClassifier(random_state=RANDOM_STATE), False),
}
results = [evaluate(f, s, y, name) for name, (f, s) in MODELS.items()]
display(
    pd.DataFrame(results).drop(columns=[c for c in results[0] if c.startswith("_")])
)

dummy = results[0]
# 🔴 더미 PR-AUC는 유병률과 같아야 한다 — 어긋나면 지표 계산이 틀린 것이다.
assert abs(dummy["_ap"] - dummy["_prev"]) < 1e-3, (
    f"더미 PR-AUC {dummy['_ap']} != 유병률 {dummy['_prev']}"
)
print("✅ 더미 PR-AUC == 유병률 — 지표 계산 경로 정상")

In [ ]:
# 🔴 음성 대조 — 라벨을 섞으면 AUC가 0.5(더미) 수준으로 떨어져야 한다.
#    안 떨어지면 누수이거나 평가 버그다. 좋은 지표라는 성공 신호를 의심할 유일한 수단.
rng = np.random.default_rng(0)
y_shuffled = rng.permutation(y)
best = max(results[1:], key=lambda r: r["_auc"])
# 신호를 주장하는 모델과 같은 모델로 대조한다 — 다른 모델이면 그 모델에만 있는
# 누수 경로(비선형이 날짜 구조를 외우는 경우 등)를 보지 못한다.
best_factory, best_scale = MODELS[best["모델"]]
shuf = evaluate(best_factory, best_scale, y_shuffled, f"{best['모델']}(라벨 셔플)")

print(f"최고 모델 {best['모델']}: ROC-AUC {best['ROC-AUC']} / 일별 {best['일별 AUC']}")
print(f"{shuf['모델']:<22}: ROC-AUC {shuf['ROC-AUC']} / 일별 {shuf['일별 AUC']}")
if abs(shuf["_auc"] - 0.5) > 0.01:
    print("🔴 셔플했는데 0.5에서 벗어난다 — 누수 또는 평가 버그.")
    print("   아래 수치는 신뢰할 수 없다.")
else:
    print("→ 셔플 시 신호 없음. 파이프라인은 라벨을 통해서만 배운다.")
print()
gap = best["_auc"] - best["_dauc"]
print(f"풀링 - 일별 AUC 차이 {gap:+.4f}")
if gap > 0.02:
    print("→ 성능의 상당 부분이 **날짜(시장 방향)** 를 맞힌 것이다.")
    print("  거시·시장 공통 피처의 몫이다.")

In [ ]:
import json
import shutil
import subprocess
from datetime import datetime, timezone

import joblib
import sklearn

# 🔴 산출물은 **저장소 밖**에 둔다. 쓰기 경로를 막는 hook은 없고
#    Jupyter 커널은 하네스 밖이라 아래 검사가 커널 축의 유일한 방어다
#    (03과 같은 패턴).
git_bin = shutil.which("git")


def _git(*args, default=""):
    """git을 읽기 전용으로 부른다. 실패하면 default."""
    if not git_bin:
        return default
    out = subprocess.run(  # noqa: S603
        [git_bin, *args], cwd=Path.cwd(), capture_output=True, text=True, check=False
    )
    return out.stdout.strip() or default


_toplevel = _git("rev-parse", "--show-toplevel")
REPO_ROOT = Path(_toplevel).resolve() if _toplevel else Path.cwd().parent.resolve()
ARTIFACT_DIR = (
    Path(os.environ.get("DATA_EXTRACT_DIR", Path.home() / "extracts")).expanduser()
    / "ml"
    / SLUG
)
resolved = ARTIFACT_DIR.resolve()
if resolved == REPO_ROOT or REPO_ROOT in resolved.parents:
    msg = f"산출물 경로가 저장소 안이다: {resolved} (저장소 루트 {REPO_ROOT})"
    raise RuntimeError(msg)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 최고 모델을 전체 데이터로 다시 적합해 저장한다.
best_factory, best_scale = MODELS[best["모델"]]
final = make_pipeline(best_factory(), best_scale)
final.fit(df, y)

# 실험 추적의 실체는 이 JSON 하나다(MLflow 없음). 다시 만들 수 있을 만큼 적는다.
strip = lambda r: {k: v for k, v in r.items() if not k.startswith("_")}  # noqa: E731
metrics = {
    "task": SLUG,
    "question": "predict next-trading-day direction (label_up) from past-only features",
    "source_table": GOLD,
    "engine": f"Trino (gold) + scikit-learn {sklearn.__version__} (host)",
    "features": FEATURES,
    "n_rows": len(df),
    "n_tickers": int(df["ticker"].nunique()),
    "n_trade_dates": int(df["trade_date"].nunique()),
    "prevalence": float(prevalence),
    "random_state": RANDOM_STATE,
    "cv": (
        f"TimeSeriesSplit(n_splits={N_SPLITS}, gap={EMBARGO_DAYS})"
        " on trade_date + purge"
    ),
    "results": [strip(r) for r in results],
    "shuffle_control": strip(shuf),
    "best_model": best["모델"],
    "git_commit": _git("rev-parse", "HEAD", default="unknown"),
    "created_at": datetime.now(timezone.utc).isoformat(),
}
joblib.dump(final, ARTIFACT_DIR / "model.joblib")
(ARTIFACT_DIR / "metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2)
)
print(f"저장: {ARTIFACT_DIR}  (model.joblib, metrics.json)")

## 한계 — 비우지 않는다

1. **생존 편향은 적재 창 안에서만 해소**된다 — 그날 거래된 종목 전체를 받으므로 상장폐지 종목이
   빠지지 않지만, 적재 시작 이전 구간은 없다([`stock-forecast.md`](../docs/dataset_schema/stock-forecast.md) §선언된 공백).
   유동성 하한은 **그날의 과거 20일 값**으로 거르므로 미래 생존을 보지 않는다.
2. **기간이 짧다** — 적재 시작일 이후 약 2년이다. 한 시장 국면(regime)만 보았을 수 있다.
3. **거래비용·체결 미반영** — AUC가 더미보다 높아도 수익을 뜻하지 않는다.
4. **유동성 하한은 임의값**이다(모델 SQL의 `{% set %}`). 하한을 바꾸면 결론이 바뀔 수 있다.
5. **하이퍼파라미터 탐색이 없다.** 베이스라인이므로 기본값만 썼다.
6. **수익률 0은 false**로 들어갔다 — 비율은 위 셀이 출력한다.

## 리포트로 옮기려면

gold는 이미 있다(`stock_forecast__direction_training_h1`). 리포트(`docs/analyses/NN-stock-direction-baseline.md`)는
**이 노트북을 위→아래 1회 실행한 결과**의 `metrics.json`을 근거로, 산출 엔진(Trino + scikit-learn)을 병기해 쓴다.


In [ ]:
conn.close()
print("커넥션 종료 — Trino는 ArgoCD가 상시 운영하므로 내릴 컴퓨트가 없다.")